In [1]:
import ipynb
from ipynb.fs.full.utils import *
from ipynb.fs.full.SVM import *
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV 

To run this program, simply run all the cells in order, unless otherwise noted.

In [1]:
#Loads in patient info
patients_info= pd.read_csv('patients_info_filtered.csv',
                          dtype={'patient': 'str', 'file': 'str', 'seizure length': np.float64, 'preictal epochs': np.float64})
patients= pd.unique(patients_info['patient'])[:10]

NameError: name 'pd' is not defined

In [3]:
methods= ['svm','rf', 'logistic', 'mlp']
param_grids = [{'svc__C': [10, 100, 1000],
                'svc__gamma': [0.01, 0.001, 0.0001]},
               {'rf__n_estimators': [10, 100, 1000],
                'rf__max_features': ['sqrt', 'log2']},
               {'logit__C': [1, 0.1, 0.01, 0.001],
                'logit__l1_ratio': [0, 0.25, 0.5, 0.75, 1]},
               {'mlp__hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 100)],
                'mlp__alpha': np.logspace(-1, 1, 4)}]
models= [('svc', svm.SVC(class_weight='balanced', probability=True)),
           ('rf', RandomForestClassifier()),
           ('logit', LogisticRegression(class_weight='balanced', max_iter=1000, penalty='elasticnet', solver='saga')),
           ('mlp', MLPClassifier(max_iter=1000))]

In [4]:
#This creates the results array for the 3-class prediction case
results= pd.DataFrame(columns=
                      ['patient', 'method', 'fold', 'test_acc', 'train_acc', 'test_precision', 'train_precision',
                       'test_recall', 'train_recall', 'test_f1', 'train_f1',
                       'test_roc_auc', 'train_roc_auc', 'test_f1_ictal', 'test_f1_preictal', 
                       'test_f1_bckg', 'train_f1_ictal', 'train_f1_preictal', 'train_f1_bckg'])

In [4]:
#If you are loading previously saved data, run this cell, uncomment and run. Otherwise, this can be skipped
#results=pd.read_csv('pipeline_results_detection.csv', index_col=0)

In [ ]:
#Changing the value of predict changes the prediction window.
#For 2-class detection, set predict=0

#By default, the results are saved and updated after every patient, for every method. If the pipeline is interrupted midway, load the csv and

predict=30
num_classes=3
if predict==0:
    num_classes=2
    results= pd.DataFrame(columns=
                      ['patient', 'method', 'fold', 'test_acc', 'train_acc', 'test_precision', 'train_precision',
                       'test_recall', 'train_recall', 'test_f1', 'train_f1',
                       'test_roc_auc', 'train_roc_auc', 'test_f1_ictal', 
                       'test_f1_bckg', 'train_f1_ictal', 'train_f1_bckg'])
#The results file saves after every patient. If the pipeline is interrupted, use the previous cell to load the file.
#The "next" variable contains the starting indices for the model and patient. If you are loading a previous file, change these to the index of the next model and patient that hasn't been added yet
#For example, if results are saved up to the 5th patient for the 2nd model, set next=[1,6] to resume from where you left off
next=[0,0]

for i, model in enumerate(models):
    if i<next[0]:
        continue
    elif i==next[0]:
        patients_use=patients[next[1]:]
    else:
        patients_use=patients
    print('Method: '+methods[i])
    for patient in patients_use:
        print('Patient: '+patient)
        test_patient= patients_info.loc[np.where(patients_info['patient']==patient)]
        test_files= test_patient['file']
        x_all, y_all, skipped, metadata= load_all_data(test_files, duration=5, overlap=3, predict=predict)
        files_unique= pd.unique(metadata['File'])
        x_short= np.array([])
        y_short= np.array([])
        for file in files_unique:
            index= metadata.index[metadata['File']==file]
            #if len(np.where(y_all[index,1]==1)[0])==0:
            #    continue
            #print(metadata.loc[index])
            x_short_p, y_short_p, meta_short_p= truncate_data(x_all[index], y_all[index], metadata.loc[index], seeded=True)
            if x_short.size==0:
                x_short=x_short_p
                y_short=y_short_p
                metadata_short=meta_short_p
            else:
                x_short= np.append(x_short, x_short_p, axis=0)
                y_short= np.append(y_short, y_short_p, axis=0)
                metadata_short= pd.concat([metadata_short, meta_short_p], axis=0)
        metadata_short.reset_index(inplace=True, drop=True)
        x_features=get_features_full(x_short)
        y_fixed= np.argmax(y_short, axis=1)
        param_grid= param_grids[i]
        pipe= Pipeline([('scaler', StandardScaler()), model])
        outer_cv= GroupKFold(n_splits=5)
        inner_cv= GroupKFold(n_splits=4)
        grid = GridSearchCV(pipe, param_grid, refit = True, cv=inner_cv, n_jobs=-1)
        nested_score= cross_validate(grid, x_features, y_fixed, groups=metadata_short['File'], cv=outer_cv, params={'groups': metadata_short['File']},
                            return_estimator=True, return_indices=True, n_jobs=-1, return_train_score=True,
                            scoring=['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted', 'roc_auc_ovo'])
        print(nested_score)
        test_class_scores= np.zeros((5,num_classes))
        train_class_scores= np.zeros((5, num_classes))
        for n in np.arange(5):
            test=nested_score['indices']['test'][n]
            x_test=x_features[test]
            y_test=y_fixed[test]
            y_predict=nested_score['estimator'][n].predict(x_test)
            class_score=f1_score(y_test, y_predict, average=None)
            test_class_scores[n]=class_score
            print(class_score)
            train=nested_score['indices']['train'][n]
            x_train=x_features[train]
            y_train=y_fixed[train]
            y_train_predict=nested_score['estimator'][n].predict(x_train)
            train_score=f1_score(y_train, y_train_predict, average=None)
            train_class_scores[n]=train_score
            print(train_score)
        best_params= np.empty((5,2), 'str_')
        for n in np.arange(5):
            estimator= nested_score['estimator'][n]
            params=estimator.best_params_
            if methods[i]=='svm':
                best_params[n,0]= params['svc__C']
                best_params[n,1]= params['svc__gamma']
            elif methods[i]=='rf':
                best_params[n,0]= params['rf__n_estimators']
                best_params[n,1]= params['rf__max_features']
            elif methods[i]=='logistic':
                best_params[n,0]= params['logit__C']
                best_params[n,1]= params['logit__l1_ratio']
            elif methods[i]=='mlp':
                best_params[n,0]= str(params['mlp__hidden_layer_sizes'])
                best_params[n,1]= params['mlp__alpha']
            #best_params[n]=params
            print(params)
        result_single= pd.DataFrame(columns=results.columns)
        result_single['patient']=np.repeat(patient, 5)
        result_single['method']=np.repeat(methods[i], 5)
        result_single['fold']=np.arange(5)
        result_single['test_acc']=nested_score['test_accuracy']
        result_single['train_acc']=nested_score['train_accuracy']
        result_single['test_precision']=nested_score['test_precision_weighted']
        result_single['train_precision']=nested_score['train_precision_weighted']
        result_single['test_recall']=nested_score['test_recall_weighted']
        result_single['train_recall']=nested_score['train_recall_weighted']
        result_single['test_f1']=nested_score['test_f1_weighted']
        result_single['train_f1']=nested_score['train_f1_weighted']
        result_single['test_roc_auc']=nested_score['test_roc_auc_ovo']
        result_single['train_roc_auc']=nested_score['train_roc_auc_ovo']
        result_single['test_f1_ictal']= test_class_scores[:,0]
        result_single['train_f1_ictal']= train_class_scores[:,0]
        result_single['param_1']= best_params[:,0]
        result_single['param_2']= best_params[:,1]
        if num_classes==3:
            result_single['test_f1_preictal']= test_class_scores[:,1]
            result_single['test_f1_bckg']= test_class_scores[:,2]
            result_single['train_f1_preictal']= train_class_scores[:,1]
            result_single['train_f1_bckg']= train_class_scores[:,2]
        else:
            result_single['test_f1_bckg']= test_class_scores[:,1]
            result_single['train_f1_bckg']= train_class_scores[:,1]
        if results.size==0:
            results=result_single
        else:
            results= pd.concat([results, result_single], axis=0)
            results.reset_index(inplace=True, drop=True)
        results.to_csv('pipeline_results.csv')

Method: logistic
Patient: aaaaapks
aaaaapks


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    1.2s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


aaaaapks


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


aaaaapks


...

aaaaardk


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


aaaaardk


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


(18458, 11)
(18458, 55)
(18458, 110)
(18458, 10)
[]
